# Модуль 03 · Урок 1. Пропуски, дубликаты и стратегия очистки

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Что считается пропуском
Пропуск — не то же самое, что ноль, пустая строка или слово «нет». В pandas данные могут содержать `NaN` (традиционный маркер числовых пропусков), `None`, `pd.NA` (nullable-типы) и `NaT` (время). Для их поиска служат `isna()` и `notna()`. Важен бизнес-смысл: отсутствующая сумма оплаты не означает нулевую оплату, а неизвестный возраст нельзя бездумно заменить на 0.

In [ ]:
import pandas as pd
supplies = pd.DataFrame({'product':['папка','ручка','степлер'], 'qty':[4,None,0],'supplier':['А',None,'Б']})
print(supplies.isna().sum())
print(supplies.loc[supplies['qty'].isna()])

## 2. Удаление и заполнение
`dropna(subset=[...])` убирает строки, где не заполнены обязательные поля. `fillna(0)` допустим только там, где ноль соответствует смыслу. Для численных показателей можно использовать медиану: `df['score'].fillna(df['score'].median())`. Медиана устойчивее к большим выбросам, чем среднее, но не всегда подходит предметной области. Можно заполнять пропуски по группам: медианой внутри категории через `groupby(...).transform('median')`.

In [ ]:
scores = pd.DataFrame({'team':['red','red','blue','blue'], 'score':[10,None,4,8]})
median_per_team = scores.groupby('team')['score'].transform('median')
print(scores.assign(score_filled=scores.score.fillna(median_per_team)))

## 3. Дубликаты: одинаковые строки и одинаковые ключи
`duplicated()` по умолчанию отмечает повторные строки целиком; `duplicated(subset=['id'])` выявляет дубликаты бизнес-ключа, даже если другие поля различаются. `drop_duplicates(subset=[...],keep='first')` оставляет первое появление; `keep='last'` — последнее. Не объединяйте конфликтующие записи автоматически: два одинаковых ID с разной ценой могут означать ошибку или версии одного объекта.

In [ ]:
customers = pd.DataFrame({'email':['a@x','b@x','a@x'],'name':['Анна','Борис','Анна']})
print(customers.duplicated(subset=['email']))
print(customers.drop_duplicates(subset=['email']))

## 4. План очистки и контроль потерь
Сначала оцените число строк и пропусков, потом выберите правила, затем снова проверьте статистику. Не удаляйте строки потому, что «так проще». Если пропущено критичное поле, можно направить запись на ручную проверку. Для отчётности полезно отдельно считать число удалённых строк и заполненных значений.

**Проверьте себя:** почему `''` — не `NaN`? В каких ситуациях нельзя заполнять нулём? Чем дублирование ключа отличается от полного дубля строки?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m3_01 · missing report

Для каждого столбца получите Series с числом пропусков, отсортированную по убыванию (при совпадении порядок колонок сохраняется).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_01: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_01 = expected_for("m3_01", df=df)


In [ ]:
# m3_01: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 5 значений; имя серии: `None`.

```text
item        1
quantity    1
category    1
email       1
price       0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_01", result, _expected_m3_01)

### m3_02 · require columns

Удалите строки с пропуском хотя бы в одном из столбцов списка `required`. Другие пропуски допустимы.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.
- `required` — список обязательных колонок; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `required`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_02: используются переменные df, required

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

required = ['item', 'quantity']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_02 = expected_for("m3_02", df=df, required=required)


In [ ]:
# m3_02: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, required
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 4 строки, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
     item   price quantity category          email
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM
2     b-2                3     food        b@x.ru 
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM
15   d-4    -4.00        1      toy           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_02", result, _expected_m3_02)

### m3_03 · fill category

Получите таблицу, где пропуски в `category` заменены строкой `unknown`, а остальные значения сохранены.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_03: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_03 = expected_for("m3_03", df=df)


In [ ]:
# m3_03: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
     item   price quantity category          email
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM
2     b-2                3     food        b@x.ru 
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM
8     C-3  ошибка     None      TOY            bad
10   None       0        7  unknown               
15   d-4    -4.00        1      toy           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_03", result, _expected_m3_03)

### m3_04 · deduplicate items

Оставьте последнее вхождение каждого значения `item`, считая разные пробелы значимыми. Сохраните порядок оставшихся строк.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_04: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_04 = expected_for("m3_04", df=df)


In [ ]:
# m3_04: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
     item   price quantity category          email
2     b-2                3     food        b@x.ru 
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM
8     C-3  ошибка     None      TOY            bad
10   None       0        7     None               
15   d-4    -4.00        1      toy           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_04", result, _expected_m3_04)

### m3_05 · missing percent

Получите Series с процентом пропущенных значений по каждому столбцу от 0 до 100 (без округления).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_05: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_05 = expected_for("m3_05", df=df)


In [ ]:
# m3_05: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 5 значений; имя серии: `None`.

```text
item        16.666667
price        0.000000
quantity    16.666667
category    16.666667
email       16.666667
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_05", result, _expected_m3_05)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()